# Intermediate Fusion Solution
## ResNet-50 & ClinicalBERT

This solution uses one custom model that, based on the combination of embeddings from RestNet-50 (for image processig) and ClinicalBERT (for medical text processing), classifies a mammography as presenting a benign or malignant tumor.

In [9]:

from MultimodalBreastCancerModel import MultimodalBreastCancerModel
from DMIDMultiModalDataset import DMIDMultimodalDataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import numpy as np


import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")


def get_dataloaders(batch_size=16):
    torch.manual_seed(42)

    full_dataset = DMIDMultimodalDataset(
        metadata_path="./archive/DMID_PNG/Metadata.xlsx",
        image_dir="./archive/DMID_PNG/512/PLA_PNG/",
        reports_dir="./archive/DMID_PNG/Reports/"
    )

    total_size = len(full_dataset)

    # 80% of the dataset will be used for training, 10% for validation and 10% for testing
    train_size = int(0.8 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset, [train_size, val_size, test_size]
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

PyTorch Version: 2.5.1+cu121
CUDA Available: True


### Training phase

Chose a smaller training rate for the individual models processing the images and clinical notes, and a bigger one (1e-3) for the final layer of the classificator.

In [10]:
def run_training_phase():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Starting Training on: {device}")

    # Get only the Train and Validation loaders
    train_loader, val_loader, _ = get_dataloaders()

    model = MultimodalBreastCancerModel(num_classes=2, dropout_rate=0.3,classifier="mlp").to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam([
        {'params': model.image_extractor.parameters(), 'lr': 1e-5},
        {'params': model.text_extractor.parameters(), 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-3}
    ])

    best_val_loss = float('inf')
    num_epochs = 10

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        #loading bar for training
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [TRAIN]")

        for batch in train_bar:
            images, input_ids, attention_mask, labels = [batch[k].to(device) for k in ('image', 'input_ids', 'attention_mask', 'label')]

            optimizer.zero_grad()
            logits = model(images, input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_bar.set_postfix(loss=loss.item())


        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                images, input_ids, attention_mask, labels = [batch[k].to(device) for k in ('image', 'input_ids', 'attention_mask', 'label')]

                logits = model(images, input_ids, attention_mask)
                loss = criterion(logits, labels)
                val_loss += loss.item()

                predictions = torch.argmax(logits, dim=1)
                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = accuracy_score(all_labels, all_preds)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

        if avg_val_loss < best_val_loss:
            print(">>> New best model found! Saving 'best_breast_cancer_model.pth'...")
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_breast_cancer_model.pth")


### Final testing phase

During this phase, we will take into account the following metrics:
- accuracy
- recall
- precision
- F1 score

In [11]:
def run_testing_phase():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n--- INITIATING FINAL TEST ON UNSEEN DATA ---")

    # gets only the test loader
    _, _, test_loader = get_dataloaders()


    model = MultimodalBreastCancerModel(num_classes=2,classifier="mlp").to(device)

    try:
        model.load_state_dict(torch.load("best_breast_cancer_model.pth"))
        print("Successfully loaded 'best_breast_cancer_model.pth'")
    except FileNotFoundError:
        print("Error: Could not find the saved model. You must run run_training_phase() first!")
        return

    model.eval()
    test_preds, test_labels = [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            images, input_ids, attention_mask, labels = [batch[k].to(device) for k in ('image', 'input_ids', 'attention_mask', 'label')]

            logits = model(images, input_ids, attention_mask)
            predictions = torch.argmax(logits, dim=1)

            test_preds.extend(predictions.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    # Calculate final metrics
    acc = accuracy_score(test_labels, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='binary')

    print(f"\nFINAL TEST RESULTS:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")

### Running the model phase (train + test)

In [12]:
%%time

if __name__ == "__main__":
    run_training_phase()
    run_testing_phase()

Starting Training on: cuda
Dataset loaded: Found 404 valid patients with images.


Epoch 1/10 [TRAIN]: 100%|██████████| 21/21 [00:54<00:00,  2.59s/it, loss=1.37] 


Epoch 1 | Train Loss: 0.6631 | Val Loss: 0.3846 | Val Acc: 0.8750
>>> New best model found! Saving 'best_breast_cancer_model.pth'...


Epoch 2/10 [TRAIN]: 100%|██████████| 21/21 [00:56<00:00,  2.70s/it, loss=0.0419]


Epoch 2 | Train Loss: 0.2597 | Val Loss: 0.2290 | Val Acc: 0.9000
>>> New best model found! Saving 'best_breast_cancer_model.pth'...


...

Epoch 10 | Train Loss: 0.1232 | Val Loss: 0.2810 | Val Acc: 0.8750

--- INITIATING FINAL TEST ON UNSEEN DATA ---
Dataset loaded: Found 404 valid patients with images.


C:\Users\maria\AppData\Local\Temp\ipykernel_11028\1489245766.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_breast_cancer_model.

Successfully loaded 'best_breast_cancer_model.pth'


Testing: 100%|██████████| 3/3 [00:00<00:00,  3.14it/s]


FINAL TEST RESULTS:
Accuracy:  0.9268
Precision: 0.8667
Recall:    0.9286
F1-Score:  0.8966
CPU times: total: 5min 51s
Wall time: 9min 39s
